# Chatbot and RAG Evaluation

  Retrival Augmented Generation(RAG) is a technique that enhances the Large Language Model(LLMs) by providing them with relavent external knowledge. It becomes one of the most widely used approaches for building LLM applications

 How to evaluate the RAG applications using LangSmith
 1.How to create test datasets
 2.How to run your RAG applications on those datasets\n
 3.How to measure your application's performance using different evaluation metrics\n


### Overview

A typical RAG evaluation workflow consists of three main steps:

1.Creating a dataset with questions and there expected answer
2.Running your RAG applications on those questions
3.Using evaluators to measure how well your pplicaton performed, looking at factors like:

Answer relevance
Answer accuracy'
Retrieval quality

We will create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog post


### ChatBot Evaluation

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGSMITH_API_KEY']=os.getenv('LANGSMITH_API_KEY')
os.environ['OPENROUTER_API_KEY']=os.getenv('OPENROUTER_API_KEY')
os.environ['LANGSMITH_TRACING']="true"



In [ ]:
## create the datapoints

from langsmith import Client
client=Client()

dataset_name="Simple Evaluation Chatbot"
dataset=client.create_dataset(dataset_name=dataset_name)


client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "LangChain is a framework for building LLM applications"}
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "LangSmith is a platform for building and evaluating LLM applications"}
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "OpenAI is a company that builds AI models"}
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "Google is a company that builds search engines"}
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "Mistral is a company that builds AI models"}
        },
    ]

)



{'example_ids': ['9f4e397c-7fe9-41f8-a9bb-783bc9bbd1d2',
  'a5d357dc-81ae-45ec-a4ca-eeb233dfd02b',
  '1af0af43-4dca-420c-a627-628d2a343f5c',
  '3387ca69-7fcd-4cab-a8b1-bfa543e8e86f',
  '53b3b3ee-2f5d-4a8c-a698-d324cf437330'],
 'count': 5,
 'as_of': '2026-10-10T08:57:05.289322395Z'}

### Define the Metrics(LLM as a judge)

In [19]:
import openai
from langsmith import wrappers

openai_client=wrappers.wrap_openai(openai.OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
))

eval_instructions="""you are an expert profeesor specialized in grading students answeto the questions"""

def corectness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content=f"""
    you are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT OR INCORRECT
    Grade:
    
    """
    response=openai_client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        temperature=0.0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ],
    )
    return response.choices[0].message.content.strip() == "CORRECT"

In [20]:
## Concision - check whether the actual output is less than 2x the length of the expected result

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs['response'])) < 2 * int(len(reference_outputs['answer']))

### Run Evaluation

In [21]:
default_instructions="Respond to the users question in a short and concise manner (one short sentence)"
def my_app(question: str, model: str = "nvidia/nemotron-3-super-120b-a12b:free", instructions: str=default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question}
        ],
    ).choices[0].message.content



In [22]:
# Call my_app for every datapoints
def ls_target(inputs: str)->str:
    return {"response": my_app(inputs['question'])}

In [23]:
## Run our evaluation
experimental_results=client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[corectness, concision],
    experiment_prefix="Evaluation Chatbot",
)

View the evaluation results for experiment: 'Evaluation Chatbot-e6ac2b9a' at:
https://smith.langchain.com/o/10871889-9996-4579-886b-05660b35518f/datasets/4e350f0f-4d75-4850-940d-53d4c3fab4cc/compare?selectedSessions=2a82607b-7b78-4552-b6dc-5f9d1d1d008a




5it [00:33,  6.74s/it]


### Evaluation For RAG

In [26]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
# from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[item for sublist in docs for item in sublist]


text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
)
doc_splits=text_splitter.split_documents(docs_list)

# Add the document chunks to the vector store using HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

#with langchain we can easy turn any vector store into a retrieval component
retriever=vectorstore.as_retriever(k=6)




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2965.82it/s]


In [27]:
retriever.invoke("What is prompt engineering?")

[Document(id='863fc950-6a1d-499f-8789-3f53afdf649a', metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en-us'}, page_content="Prompt Engineering | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\

In [29]:
from langchain.chat_models import init_chat_model
llm=init_chat_model(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.0,
)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F1BD06ED50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F1B73B8EC0>, root_client=<openai.OpenAI object at 0x000001F1BD06EAD0>, root_async_client=<openai.AsyncOpenAI object at 0x000001F1BD06F250>, model_name='nvidia/nemotron-3-super-120b-a12b:free', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', stream_chunk_timeout=120.0)

In [31]:
from langsmith import traceable

@traceable
def rag_bot(question: str)->dict:
    docs=retriever.invoke(question)
    docs_string= "\n".join([doc.page_content for doc in docs])

    instructions=f"""you are helpful assistant who is good at analysing source information and answering question. Use the following source docuemnts to answer. if you don't know the answer just ay that you don't know. Use three sentences and keep it short and concise

    Documents:{docs_string}
    """
    ai_mssg=llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    return {"answer": ai_mssg.content, "documents": docs}

In [32]:
rag_bot("What is prompt engineering?")

{'answer': 'Prompt engineering, also called in‑context prompting, involves crafting inputs to guide a language model’s behavior toward desired outputs without changing its weights. It relies on techniques such as zero‑shot, few‑shot, chain‑of‑thought, and self‑consistency sampling to shape the model’s reasoning and generation. The effectiveness of these prompts varies across models and tasks, requiring experimentation and heuristic tuning.',
 'documents': [Document(id='863fc950-6a1d-499f-8789-3f53afdf649a', metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nT

### Dataset

In [33]:
from langsmith import Client

client=Client()

examples=[
    {
        "inputs": {"question": "How does ReAct agent use self-reflection?"},
        "outputs": {"answer": "ReAct agents use self-reflection to improve their performance."}
    },
    {
            "inputs": {"question": "What are the types of biases that can araise with few shot prompting?"},
            "outputs": {"answer": """The biases that can arise with few shot prompting include: (1) Mapping bias: This occurs when the prompt is biased towards a specific mapping of input to output. (2) Label bias: This occurs when the prompt is biased towards a specific label or outcome. (3) Data selection bias: This occurs when the prompt is biased towards a specific data set or subset. (4) Data distribution bias: This occurs when the prompt is biased towards a specific data distribution. (5) Data representation bias: This occurs when the prompt is biased towards a specific data representation. (6) Data quality bias: This occurs when the prompt is biased towards a specific data quality. """}
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks on LLMs?"},
        "outputs": {"answer": """Five types of adversarial attacks on LLMs are: (1) In-context adversarial attacks: These attacks involve modifying the input prompt to cause the model to make a mistake. (2) Data poisoning attacks: These attacks involve modifying the training data to cause the model to make a mistake. (3) Model poisoning attacks: These attacks involve modifying the model to cause it to make a mistake. (4) Data poisoning attacks: These attacks involve modifying the training data to cause the model to make a mistake. (5) Data poisoning attacks: These attacks involve modifying the training data to cause the model to make a mistake. """}
       
    }
    
]

#create a dataset and example in Langsmith
dataset_name="RAG Test Evaluation"
dataset=client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples,
)



{'example_ids': ['90144fb4-a30e-4993-9d61-3b1db1138480',
  '494e1ddd-50f7-4566-b0f9-1d8ab8983de8',
  'ffb3b0f5-17ed-437f-a26a-bb976885d6e1'],
 'count': 3,
 'as_of': '2026-10-10T13:49:57.750628779Z'}

### Evaluators or Metrics

### 1. Correctness: Response vs reference answer

##### Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer.
##### Mode: Requires a ground truth(reference) answer supplied though a datset.
##### Evaluators: Use LLM-as-judge to access the corect answers.

In [ ]:
from typing_extensions import TypedDict, Annotated

## Correctness Output schema

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str,..., "Explanation your Reasoning for the score"]
    correct: Annotated[bool,..., "True if the answer is correct, False otherwise"]

correctness_prompt="""You are a teacher grading a quiz
  You will be given a QUESTION, the GROUND-TRUTH (correct) ANSWER, and a STUDENT'S ANSWER.

  Here is the grade criteria to follow:
  (1) Grade the student's answer based ONLY on their factual accuracy relative to the ground truth answer.
  (2) Ensure that the student's answer does not contain any conflicting statements.
  (3) It is OK is the student answer contains more information that the ground-truth answer, as long as it is factually correct.

  Correctness grade:
  A correctness value True means that the student's answer meet all the criteria above.
  A correctness value False means that the student's answer does not meet all the criteria above.

  Explain your reasoning in step by step manner to ensure your reasoning and conclusion are correct.

  Avoid simply stating the correct answer at the outset.
"""
from langchain_openai import ChatOpenAI
grader_llm=ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
).with_structured_output(CorrectnessGrade,method="json_schema", strict=True)

def correctness(inputs:dict, outputs:dict, reference_outputs:dict)-> bool:
    """An evaluator for RAG answer accuracy"""
    answer=f"""
    QUESTION: {inputs['question']}
    GROUND-TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT'S ANSWER: {outputs['answer']}
    """
    grade=grader_llm.invoke([
        {"role": "system", "content": correctness_prompt},
        {"role": "user", "content": answer}
    ])
    return grade["correct"]






### Relevance: Response vs Input

The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a refeence answer we can't grade the accuracy, but we still grade the relevance--as in, did the model ddress the user's question or not.

In [36]:
from typing_extensions import TypedDict, Annotated

#Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ...,"Explanation your Reasoning for the score"]
    relevant: Annotated[bool, ...,"Provide the score on whether the answer address the question"]

relevance_prompt="""You are a teacher grading a quiz
  You will be given a QUESTION,and a STUDENT'S ANSWER.

  Here is the grade criteria to follow:
  (1) Ensure the STUDENT ANSWER is conscise and relevant to the question.
  (2) Ensure STUDENT ANSWER help to answer the QUESTION.

  Relevance grade:
  A relevance value True means that the student's answer meet all the criteria above.
  A relevance value False means that the student's answer does not meet all the criteria above.

  Explain your reasoning in step by step manner to ensure your reasoning and conclusions are correct.

  Avoid simply stating the correct answer at the outset.
"""
from langchain_openai import ChatOpenAI
relevance_llm=ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
).with_structured_output(RelevanceGrade,method="json_schema", strict=True)

def relevance(inputs:dict, outputs:dict, reference_outputs:dict)-> bool:
    """An evaluator for RAG answer accuracy"""
    answer=f"""
    QUESTION: {inputs['question']}
    STUDENT'S ANSWER: {outputs['answer']}
    """
    grade=relevance_llm.invoke([
        {"role": "system", "content": relevance_prompt},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]






### Groundedness: Response vs  retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved docs.

In [35]:
from typing_extensions import TypedDict, Annotated

## Grade Output schema

class GroundedGrade(TypedDict):
    explanation: Annotated[str,..., "Explanation your Reasoning for the score"]
    grounded: Annotated[bool,..., "Provide the score if the answer hallucinates from the documnets"]

grounded_prompt="""You are a teacher grading a quiz
  You will be given FACTS and a STUDENT ANSWER.

  Here is the grade criteria to follow:
  (1) Ensure the STUDENT ANSWER is grounded in the FACTS.
  (2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

  Groundedness grade:
  A groundedness value True means that the student's answer meet all the criteria above.
  A groundedness value False means that the student's answer does not meet all the criteria above.

  Explain your reasoning in step by step manner to ensure your reasoning and conclusion are correct.

  Avoid simply stating the correct answer at the outset.
"""
from langchain_openai import ChatOpenAI
groundedness_llm=ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
).with_structured_output(GroundedGrade,method="json_schema", strict=True)

def groundedness(inputs:dict, outputs:dict, reference_outputs:dict)-> bool:
    """An evaluator for RAG answer accuracy"""
    answer=f"""
    QUESTION: {inputs['question']}
    GROUND-TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT'S ANSWER: {outputs['answer']}
    """
    grade=groundedness_llm.invoke([
        {"role": "system", "content": grounded_prompt },
        {"role": "user", "content": answer}
    ])
    return grade["grounded"]






### Retrieval Relevance: Retrieved docs vs input

In [38]:
from typing_extensions import TypedDict, Annotated

## Grade Output schema

class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str,..., "Explanation your Reasoning for the score"]
    relevant: Annotated[bool,..., "True if the retrieved docs are relevant to the question, False otherwise"]

retrieval_relevance_prompt="""You are a teacher grading a quiz
  You will be given a QUESTION and set of FACTS provided by the student.

  Here is the grade criteria to follow:
  (1) Your goal is to identify FACTS that are completely unrelated to the question.
  (2)If the facts contain ANY keywords or semantic meaning elated to the question, consider them relevant.
  (3)It is OK if the FACTS have SOME information that is unrelevent to the question as long as (2) is met.

  Retrieval Relevance grade:
  A retrieval relevance value True means that the student's answer meet all the criteria above.
  A retrieval relevance value False means that the student's answer does not meet all the criteria above.

  Explain your reasoning in step by step manner to ensure your reasoning and conclusion are correct.

  Avoid simply stating the correct answer at the outset.
"""
from langchain_openai import ChatOpenAI
retrieval_relevance_llm=ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
).with_structured_output(RetrievalRelevanceGrade,method="json_schema", strict=True)

def retrieval_relevance(inputs:dict, outputs:dict, reference_outputs:dict)-> bool:
    """An evaluator for RAG answer accuracy"""
    answer=f"""
    QUESTION: {inputs['question']}
    GROUND-TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT'S ANSWER: {outputs['answer']}
    """
    grade=retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_prompt },
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]






### Run the Evaluation

In [41]:
def target(inputs:dict)->dict:
    return rag_bot(inputs['question'])

experimental_results=client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness,relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version":"LCEL context"},

)
# Explore results locally as a dataframe if you have pandas installed
experimental_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-5380ab93' at:
https://smith.langchain.com/o/10871889-9996-4579-886b-05660b35518f/datasets/aa24d4a2-4884-4fec-83d1-a6d9df78d836/compare?selectedSessions=71a01e76-52f6-47be-a8c4-7cf215cc9c82




3it [00:46, 15.58s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can araise w...,Few‑shot prompting can suffer from demonstrati...,[page_content='References#\n[1] Zhao et al. “C...,None,The biases that can arise with few shot prompt...,True,True,True,True,6.561030,494e1ddd-50f7-4566-b0f9-1d8ab8983de8,01a12684-ec0e-73f2-9b2b-5fe497a421cb
1,How does ReAct agent use self-reflection?,The ReAct agent augments each action with a se...,[page_content='In both experiments on knowledg...,None,ReAct agents use self-reflection to improve th...,True,True,True,True,1.910629,90144fb4-a30e-4993-9d61-3b1db1138480,01a12685-38a0-7f71-bf6e-9e4d03b5325a
2,What are five types of adversarial attacks on ...,The five types of adversarial attacks on LLMs ...,"[page_content='Or\n@article{weng2023attack,\n ...",None,Five types of adversarial attacks on LLMs are:...,False,True,True,True,4.116498,ffb3b0f5-17ed-437f-a26a-bb976885d6e1,01a12685-6f30-7a52-9429-9cf9d1c156ce
